# Forecasting German Electricity Demand - A Kernel-Method Perspective (Version 3)

This study models the German national electricity load and projects it two years
ahead on a weekly grid. It walks through data preparation, exploratory and
seasonality analysis, stationarity testing, a family of naive baselines, a **SARIMA**
model and its exogenous extension **SARIMAX**, a **Support Vector Regression** with an
RBF kernel as the feature-based learner, and finally an hourly LSTM. Each model is
scored on the same 104-week hold-out with RMSE, MAE and MAPE.

**Design notes specific to this version**

- The feature-based stage uses **kernel Support Vector Regression** rather than a
  tree ensemble. Kernels demand scaled inputs and a tuned `C`/`gamma`/`epsilon`
  triple, so the modelling pipeline is built around a scaler and a cross-validated
  grid search on a time-aware split.
- Annual seasonality is encoded with **sine/cosine calendar features** so the kernel
  sees a smooth cyclical signal instead of a raw week number.
- Seasonality is separated with **STL** rather than a classical moving-average
  decomposition.
- Because kernel machines expose no built-in feature ranking, importance is measured
  by **permutation** on the hold-out.
- The SARIMA order is chosen **residual-adequacy first**: candidate `(p, q)` orders
  are refit at `d in {0, 1}` (AIC is only comparable within a fixed `d`), filtered to
  those whose residuals are white noise (Ljung-Box `p > 0.05`), and the smallest such
  `d` with the lowest-AIC, most parsimonious order is kept; the seasonal order is
  chosen the same way. A low AIC never overrides a failed residual check.
- The machine-learning forecast is generated recursively (true multi-step) and, for
  reference only, in a one-step-ahead mode that consumes the observed previous week.

## Step 1 - Toolkit and conventions

The opening cell fixes the plotting theme, a compact colour key, the random seed and
a single scoring routine that returns a named triple of error measures. Keeping the
metric logic in one place means every model below is judged identically.

In [ ]:
!pip -q install holidays
import warnings
warnings.filterwarnings('ignore')

from collections import namedtuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- plotting identity ------------------------------------------------------
plt.style.use('seaborn-v0_8-whitegrid')
PAL = {
    'cyan': '#118ab2',
    'pink': '#ef476f',
    'mint': '#06d6a0',
    'navy': '#073b4c',
    'gold': '#ffd166',
    'grey': '#8d99ae',
}
plt.rcParams['axes.prop_cycle'] = plt.cycler(
    color=[PAL['cyan'], PAL['pink'], PAL['mint'], PAL['gold'], PAL['navy'], PAL['grey']])
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10

RNG = 7
np.random.seed(RNG)

Scores = namedtuple('Scores', ['rmse', 'mae', 'mape'])


def score(y_obs, y_hat):
    """Return an (rmse, mae, mape) named triple for a forecast."""
    y_obs = np.asarray(y_obs, float)
    y_hat = np.asarray(y_hat, float)
    gap = y_obs - y_hat
    return Scores(
        rmse=float(np.sqrt(np.square(gap).mean())),
        mae=float(np.abs(gap).mean()),
        mape=float(np.abs(gap / y_obs).mean() * 100.0),
    )


def rmse_only(y_obs, y_hat):
    """Convenience RMSE for the deep-learning section."""
    y_obs = np.asarray(y_obs, float)
    y_hat = np.asarray(y_hat, float)
    return float(np.sqrt(np.square(y_obs - y_hat).mean()))

## Step 2 - Acquire and bin the series

We read the Open Power System Data 60-minute file, keep the German actual-load column,
clip to the 2015-2020 window and roll the hourly readings up to daily and weekly means.
Weekly means are the primary modelling grid; the hourly signal is kept aside for the
neural network in Step 9.

In [ ]:
import os
# Portable data location: try the Kaggle mount first, then a couple of local paths, so
# the notebook also runs off-Kaggle. Point DATA_LOCATIONS at your copy of the OPSD
# 60-minute file (https://data.open-power-system-data.org/time_series/) if it lives
# somewhere else.
DATA_LOCATIONS = [
    '/kaggle/input/datasets/rishiande/german/opsd_60min_raw.csv',
    'opsd_60min_raw.csv',
    'data/opsd_60min_raw.csv',
]
DATA_FILE = next((p for p in DATA_LOCATIONS if os.path.exists(p)), None)
if DATA_FILE is None:
    raise FileNotFoundError(
        'opsd_60min_raw.csv was not found. Attach the OPSD 60-minute dataset to the '
        'notebook or add its path to DATA_LOCATIONS.')
frame_60min = pd.read_csv(DATA_FILE, parse_dates=['utc_timestamp'], index_col='utc_timestamp')
print(f'Hourly rows in file: {frame_60min.shape[0]:,}  (source: {DATA_FILE})')

In [ ]:
LOAD_KEY = 'DE_load_actual_entsoe_transparency'
power_60min = frame_60min[[LOAD_KEY]].rename(columns={LOAD_KEY: 'mw'}).copy()
# The load column ends 2020-09-30; the 2020-10-31 upper bound is only a harmless slice
# ceiling (.loc stops at the last available row) - not an accidental mid-series cut-off.
power_60min = power_60min.loc['2015-01-01':'2020-10-31'].dropna()
print('Window :', power_60min.index.min(), 'to', power_60min.index.max())
print('Hourly observations retained:', f'{power_60min.shape[0]:,}')

In [ ]:
mw_daily = power_60min['mw'].resample('D').mean()
demand_weekly = power_60min['mw'].resample('W').mean()
print('Weekly observations :', demand_weekly.size)
print('Contains gaps        :', bool(demand_weekly.isna().any()))

In [ ]:
overview = pd.DataFrame({
    'daily': mw_daily.describe(),
    'weekly': demand_weekly.describe(),
}).round(1)
print(overview.to_string())

## Step 3 - Exploratory analysis and seasonality

Two views: the daily series against its quarterly rolling mean to expose the trend, and
a **year-overlay** of the weekly profile, where each calendar year is drawn as its own
line against the week-of-year axis. The overlay makes the repeated winter peak / summer
trough obvious and shows how little the annual shape moves from year to year (apart from
the visibly depressed 2020 line).

In [ ]:
fig, (upper, lower) = plt.subplots(1, 2, figsize=(15, 5))

upper.plot(mw_daily.index, mw_daily, color=PAL['grey'], lw=0.6, alpha=0.8)
upper.plot(mw_daily.rolling(90, center=True).mean(), color=PAL['navy'], lw=2.2,
           label='90-day rolling mean')
upper.set_title('Daily load with trend')
upper.set_ylabel('MW')
upper.legend(fontsize=8)

calendar = pd.DataFrame({'mw': demand_weekly})
calendar['yr'] = calendar.index.year
calendar['week'] = calendar.index.isocalendar().week.astype(int)
palette_cycle = [PAL['cyan'], PAL['pink'], PAL['mint'], PAL['gold'], PAL['navy'], PAL['grey']]
for (yr, chunk), col in zip(calendar.groupby('yr'), palette_cycle):
    lw = 3.0 if yr == 2020 else 1.3
    lower.plot(chunk['week'], chunk['mw'], color=col, lw=lw, label=str(yr))
lower.set_title('Weekly profile overlaid by year')
lower.set_xlabel('Week of year')
lower.set_ylabel('MW')
lower.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

To separate the components numerically we apply **STL** (Seasonal-Trend decomposition
using Loess). STL is robust to outliers and, unlike a centred moving average, returns a
full-length trend and residual with no truncation at the ends of the sample.

In [ ]:
from statsmodels.tsa.seasonal import STL

stl_fit = STL(demand_weekly, period=52, robust=True).fit()
comp = {'Trend': (stl_fit.trend, PAL['navy']),
        'Seasonal': (stl_fit.seasonal, PAL['cyan']),
        'Remainder': (stl_fit.resid, PAL['pink'])}

fig, panels = plt.subplots(3, 1, figsize=(13, 7), sharex=True)
panels[0].plot(demand_weekly.index, demand_weekly, color=PAL['grey'], lw=1.0, label='Observed')
panels[0].plot(stl_fit.trend.index, stl_fit.trend, color=PAL['navy'], lw=2.0, label='STL trend')
panels[0].legend(loc='upper right', fontsize=8)
panels[0].set_ylabel('MW')
for ax, (lab, (series_c, col)) in zip(panels[1:], list(comp.items())[1:]):
    ax.axhline(0, color=PAL['grey'], lw=0.7)
    ax.plot(series_c.index, series_c, color=col, lw=1.1)
    ax.set_ylabel(lab)
panels[-1].set_xlabel('Date')
panels[0].set_title('STL decomposition of weekly demand')
plt.tight_layout()
plt.show()
print('Seasonal strength (1 - var(resid)/var(resid+seasonal)):',
      round(max(0, 1 - stl_fit.resid.var() / (stl_fit.resid + stl_fit.seasonal).var()), 3))

## Step 4 - Stationarity battery

Two hypothesis tests with opposite nulls (ADF against a unit root, KPSS against
stationarity) plus correlogram inspection guide the differencing choice for the SARIMA
model.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf


def stationarity_line(series, label):
    series = series.dropna()
    adf_p = adfuller(series)[1]
    kpss_p = kpss(series, regression='c', nlags='auto')[1]
    verdict_adf = 'stationary' if adf_p <= 0.05 else 'has unit root'
    verdict_kpss = 'non-stationary' if kpss_p < 0.05 else 'stationary'
    print(f'{label:32s} | ADF p={adf_p:6.4f} ({verdict_adf:13s}) | '
          f'KPSS p={kpss_p:6.4f} ({verdict_kpss})')


print('Stationarity summary')
print('-' * 92)
stationarity_line(demand_weekly, 'level')
stationarity_line(demand_weekly.diff(), 'first difference')
stationarity_line(demand_weekly.diff(52), 'seasonal difference (52)')

In [ ]:
fig, grid = plt.subplots(2, 2, figsize=(13, 7))
plot_acf(demand_weekly.dropna(), ax=grid[0, 0], lags=104, color=PAL['cyan'])
grid[0, 0].set_title('ACF | level')
plot_acf(demand_weekly.diff().dropna(), ax=grid[0, 1], lags=104, color=PAL['pink'])
grid[0, 1].set_title('ACF | first difference')
plot_pacf(demand_weekly.dropna(), ax=grid[1, 0], lags=52, method='ywm', color=PAL['cyan'])
grid[1, 0].set_title('PACF | level')
plot_pacf(demand_weekly.diff().dropna(), ax=grid[1, 1], lags=52, method='ywm', color=PAL['pink'])
grid[1, 1].set_title('PACF | first difference')
plt.tight_layout()
plt.show()

**Differencing verdict.** ADF rejects the unit-root null at the level while KPSS does
not flag non-stationarity, so the level is close to mean-stationary; however the slowly
decaying ACF and the lag-52 spike reveal the annual cycle. For the seasonal model we
therefore combine one seasonal difference (`D = 1` at `s = 52`) to remove the yearly
wave with one ordinary difference (`d = 1`) to absorb residual drift. A second ordinary
difference is unjustified - the series is already stationary after one - and would only
over-difference, a point the AIC search re-confirms below.

## Step 5 - Benchmark forecasts

The last 104 weeks are held out. Four reference forecasts are produced over that
horizon: a flat historical average, a carried-forward last value, a seasonal replay of
the previous year, and a straight drift line. These set the bar every later model must
clear.

In [ ]:
TEST_WEEKS = 104
insample = demand_weekly.iloc[:-TEST_WEEKS]
holdout = demand_weekly.iloc[-TEST_WEEKS:]
when = holdout.index
print(f'In-sample weeks : {insample.size:3d}  (ends {insample.index[-1].date()})')
print(f'Hold-out weeks  : {holdout.size:3d}  (ends {holdout.index[-1].date()})')

In [ ]:
YEAR = 52

avg_fc = pd.Series(insample.mean(), index=when)
hold_fc = pd.Series(insample.iloc[-1], index=when)
season_block = insample.iloc[-YEAR:].to_numpy()
snaive_fc = pd.Series(season_block[np.arange(TEST_WEEKS) % YEAR], index=when)
gradient = (insample.iloc[-1] - insample.iloc[0]) / (insample.size - 1)
drift_fc = pd.Series(insample.iloc[-1] + gradient * np.arange(1, TEST_WEEKS + 1), index=when)

benches = {'Average': avg_fc, 'Last value': hold_fc,
           'Seasonal replay': snaive_fc, 'Drift': drift_fc}
for tag, path in benches.items():
    s = score(holdout, path)
    print(f'{tag:16s} RMSE={s.rmse:8.1f}  MAE={s.mae:8.1f}  MAPE={s.mape:5.2f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(when, holdout, color=PAL['navy'], lw=2.5, label='Observed')
ax.plot(when, snaive_fc, color=PAL['pink'], lw=1.8, label='Seasonal replay')
ax.plot(when, avg_fc, color=PAL['cyan'], lw=1.3, ls=':', label='Average')
ax.plot(when, drift_fc, color=PAL['mint'], lw=1.3, ls='--', label='Drift')
ax.plot(when, hold_fc, color=PAL['gold'], lw=1.3, ls='-.', label='Last value')
ax.set_title('Benchmark forecasts over the hold-out')
ax.set_ylabel('MW')
ax.legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

## Step 6 - SARIMA

The specification loops over `p in [0,6]`, `d in [0,2]`, `q in [0,6]` (147 orders).
Because the likelihood - and therefore AIC - is computed on the `d`-times-differenced
series, AIC cannot be compared across different `d`. The workflow screens all orders
quickly, fixes `d = 1` from the stationarity evidence, refits the strongest `d = 1`
candidates exactly so their AIC shares one basis, applies a parsimony tie-break, then
searches a small seasonal grid.

In [ ]:
import itertools
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from joblib import Parallel, delayed

PERIOD = 52
grid_orders = list(itertools.product(range(7), range(3), range(7)))
BASE_SEASONAL = (1, 1, 1, PERIOD)
SS_OFF = dict(enforce_stationarity=False, enforce_invertibility=False)
print('Orders to screen:', len(grid_orders))


def scan_aic(order, y):
    """Quick simple-differencing fit; AIC valid only for ranking within one d."""
    try:
        res = SARIMAX(y, order=order, seasonal_order=BASE_SEASONAL, **SS_OFF,
                      simple_differencing=True).fit(disp=False, method='lbfgs', maxiter=45)
        return {'order': order, 'aic': res.aic, 'd': order[1],
                'ok': bool((res.mle_retvals or {}).get('converged', False))}
    except Exception:
        return {'order': order, 'd': order[1], 'aic': np.inf, 'ok': False}


scan = pd.DataFrame(Parallel(n_jobs=-1)(delayed(scan_aic)(o, insample) for o in grid_orders))
cheapest = scan.sort_values('aic').iloc[0]
print('Raw grid minimum:', tuple(cheapest['order']),
      f"(AIC={cheapest['aic']:.2f}, d={int(cheapest['order'][1])}) - not comparable across d")

In [ ]:
# AIC alone is not a valid selection criterion here: a low-AIC order whose residuals
# are still autocorrelated fails the Part 3 diagnostic requirement. We therefore refit
# candidate (p, q) orders at BOTH d = 0 and d = 1 (AIC is only comparable within a fixed
# d), record a Ljung-Box p-value for each, and let the next cell pick on residual
# adequacy first. Screening two d values also stops d from being hard-coded.
CAND_D = (0, 1)
ADEQUATE_P = 0.05          # Ljung-Box p above this  => residuals indistinguishable from white noise

def trim_resid(fit_obj, order, seasonal):
    """Standardized residuals with the state-space warm-up dropped."""
    skip = order[1] + seasonal[1] * seasonal[3]
    try:
        block = np.asarray(fit_obj.standardized_forecasts_error)
        block = block[0] if block.ndim > 1 else block
    except Exception:
        block = np.asarray(fit_obj.resid)
    tidy = pd.Series(block).replace([np.inf, -np.inf], np.nan)
    return tidy.iloc[skip:].dropna()

def fit_sarimax(order, seasonal, y, exog=None):
    """Exact ML fit plus a single Ljung-Box p-value on the trimmed residuals."""
    res = SARIMAX(y, exog=exog, order=order, seasonal_order=seasonal, **SS_OFF
                  ).fit(disp=False, method='lbfgs', maxiter=320)
    resid = trim_resid(res, order, seasonal)
    box = acorr_ljungbox(resid, lags=[min(20, max(1, len(resid) - 2))], return_df=True)
    return res, bool((res.mle_retvals or {}).get('converged', False)), box['lb_pvalue'].iloc[0]

def refit_pool(d_value, n_top=9):
    """Exact refits of the top-AIC (p, q) candidates that share this d."""
    orders = (scan[(scan['d'] == d_value) & scan['ok']].sort_values('aic')
              .head(n_top)['order'].tolist())
    if not orders:
        orders = scan[scan['d'] == d_value].sort_values('aic').head(n_top)['order'].tolist()
    if not orders:
        orders = [(1, d_value, 1), (0, d_value, 1)]
    out = []
    for order in orders:
        try:
            res, converged, lb = fit_sarimax(order, BASE_SEASONAL, insample)
            out.append({'order': order, 'aic': res.aic, 'bic': res.bic,
                        'converged': converged, 'lb_p': round(lb, 4)})
        except Exception:
            out.append({'order': order, 'aic': np.inf, 'bic': np.inf,
                        'converged': False, 'lb_p': np.nan})
    tbl = pd.DataFrame(out)
    tbl = tbl[np.isfinite(tbl['aic'])].sort_values('aic').reset_index(drop=True)
    return tbl

refit_by_d = {}
for d_value in CAND_D:
    tbl = refit_pool(d_value)
    refit_by_d[d_value] = tbl
    print(f'Exact refits at d={d_value} (AIC comparable within this d):')
    print(tbl.to_string(index=False) if not tbl.empty else '  (no convergent fit)')
    print()

In [ ]:
# Selection rule (permanent fix):
#   1. residual adequacy first  - keep only orders with white-noise residuals (lb_p > 0.05);
#   2. smallest such d          - AIC is not comparable across d, so prefer the least
#                                 differencing that already whitens the residuals
#                                 (this is also what the ADF/KPSS evidence argues for);
#   3. lowest AIC then parsimony - within the chosen d, take the min-AIC order and break
#                                 ties inside a 2-AIC band on fewest AR+MA terms
#                                 (Burnham & Anderson).
# If NO order at any d whitens the residuals, we keep the best-AIC d=1 model but say so
# openly rather than reporting a model that fails its own diagnostic.
chosen_d, pool, rule = None, None, None
for d_value in CAND_D:
    tbl = refit_by_d.get(d_value)
    if tbl is None or tbl.empty:
        continue
    ok = tbl[tbl['lb_p'] > ADEQUATE_P]
    if not ok.empty:
        chosen_d, pool = d_value, ok
        rule = f'smallest d with white-noise residuals (Ljung-Box p > {ADEQUATE_P})'
        break
if pool is None:
    fallback = refit_by_d.get(1)
    if fallback is None or fallback.empty:
        fallback = next(t for t in refit_by_d.values() if not t.empty)
    chosen_d, pool = int(fallback['order'].iloc[0][1]), fallback
    rule = 'NO white-noise candidate at any d - kept best AIC (residuals NOT white; reported honestly)'

top = pool['aic'].min()
band = pool[pool['aic'] <= top + 2.0]
if band.empty:
    band = pool.head(1)
main_order = min(band['order'], key=lambda o: (o[0] + o[2], o[0]))
refit = refit_by_d[chosen_d]          # kept for any later reference
print('Chosen differencing d :', chosen_d)
print('Selection rule        :', rule)
print('Residual-adequate set :', list(pool['order']))
print('Within 2 AIC units    :', list(band['order']))
print('Parsimonious pick     :', main_order,
      f"(lb_p={pool.loc[pool['order'] == main_order, 'lb_p'].iloc[0]})")

### Seasonal order search

With `s = 52` fixed by the annual cycle and `D = 1`, the seasonal `(P, Q)` terms are
searched over `{0, 1}`. The grid is intentionally shallow: the training window holds
fewer than four complete yearly cycles once a seasonal difference is taken, so a richer
seasonal model would chase noise. The lowest-AIC converged option wins.

In [ ]:
seasonal_pool = [(P, 1, Q, PERIOD) for P in (0, 1) for Q in (0, 1)]
srows = []
for so in seasonal_pool:
    try:
        res, converged, lb = fit_sarimax(main_order, so, insample)
        srows.append({'seasonal': so, 'aic': res.aic, 'bic': res.bic,
                      'converged': converged, 'lb_p': round(lb, 4)})
    except Exception:
        srows.append({'seasonal': so, 'aic': np.inf, 'bic': np.inf,
                      'converged': False, 'lb_p': np.nan})
seasonal_tbl = pd.DataFrame(srows)
seasonal_tbl = seasonal_tbl[np.isfinite(seasonal_tbl['aic'])].sort_values('aic').reset_index(drop=True)
if seasonal_tbl.empty:
    seasonal_tbl = pd.DataFrame([{'seasonal': BASE_SEASONAL, 'aic': np.nan, 'bic': np.nan,
                                  'converged': False, 'lb_p': np.nan}])
print(seasonal_tbl.to_string(index=False))
# Same rule as the non-seasonal step: prefer a seasonal order whose residuals are white
# noise, and take the lowest AIC among those; fall back to best AIC only if none pass.
seas_ok = seasonal_tbl[seasonal_tbl['lb_p'] > ADEQUATE_P]
if not seas_ok.empty:
    main_seasonal = seas_ok['seasonal'].iloc[0]
    print('\nResidual-adequate seasonal orders:', list(seas_ok['seasonal']))
else:
    main_seasonal = seasonal_tbl['seasonal'].iloc[0]
    print('\nNo seasonal order gave white-noise residuals; kept best AIC.')
print('Chosen seasonal order:', main_seasonal)

In [ ]:
import scipy.stats as stt

sarima_res = SARIMAX(insample, order=main_order, seasonal_order=main_seasonal, **SS_OFF
                     ).fit(disp=False, method='lbfgs', maxiter=420)

sarima_resid = trim_resid(sarima_res, main_order, main_seasonal)
normal_p = stt.shapiro(sarima_resid)[1]
box_tbl = acorr_ljungbox(sarima_resid, lags=[l for l in (10, 20, 52) if l < len(sarima_resid)],
                         return_df=True)

print('SARIMA{} x {}'.format(main_order, main_seasonal))
print('AIC={:.2f}  BIC={:.2f}'.format(sarima_res.aic, sarima_res.bic))
print('converged:', bool(sarima_res.mle_retvals.get('converged', False)))
print('Shapiro-Wilk p = {:.3f} -> {}'.format(
    normal_p, 'approx normal' if normal_p > 0.05 else 'non-normal tails'))
print('Ljung-Box:')
print(box_tbl.to_string())

### Residual diagnostics

`statsmodels` bundles four residual views into one figure - a standardized residual
trace, a kernel-density/histogram against the normal, a Q-Q plot and a correlogram.
Reading them together is the quickest way to judge whether the model has whitened the
series.

In [ ]:
diag = sarima_res.plot_diagnostics(figsize=(12, 8), lags=40)
diag.suptitle('SARIMA residual diagnostics', y=1.01)
plt.tight_layout()
plt.show()

Some autocorrelation usually lingers at longer lags and the Q-Q tails bend away from
the line, so the residuals are close to - but not exactly - white noise. The seasonal
structure is captured, yet holiday effects and the 2020 shock remain unmodelled. The
practical implication is that the Gaussian prediction intervals are approximate and may
under-cover around irregular periods, which is why the model is always read against the
seasonal-replay benchmark.

In [ ]:
sarima_out = sarima_res.get_forecast(steps=TEST_WEEKS)
sarima_fc = sarima_out.predicted_mean
sarima_fc.index = when
sarima_pi = sarima_out.conf_int(alpha=0.05)
sarima_pi.index = when

fig, ax = plt.subplots(figsize=(12.5, 5))
ax.plot(insample.index[-70:], insample.iloc[-70:], color=PAL['grey'], lw=1.1, label='Recent history')
ax.plot(when, holdout, color=PAL['navy'], lw=2.2, label='Observed')
ax.plot(when, sarima_fc, color=PAL['pink'], lw=2.0, label='SARIMA mean')
ax.fill_between(when, sarima_pi.iloc[:, 0], sarima_pi.iloc[:, 1],
                color=PAL['pink'], alpha=0.15, label='95% interval')
ax.set_title('SARIMA{} x {} forecast'.format(main_order, main_seasonal))
ax.set_ylabel('MW')
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

sarima_s = score(holdout, sarima_fc)
print(f'SARIMA  RMSE={sarima_s.rmse:.1f}  MAE={sarima_s.mae:.1f}  MAPE={sarima_s.mape:.2f}%')

## Step 7 - Temperature and calendar regressors

Weekly mean temperature for Berlin (Open-Meteo archive) stands in for national weather.
We keep the level, its square (the heating/cooling response is U-shaped), a one-week lag,
and a binary German-holiday marker. Because the hold-out is scored with *observed* future
temperature, any model that uses it is explanatory/conditional rather than truly
operational.

In [ ]:
import os
import requests
import holidays
ENDPOINT = 'https://archive-api.open-meteo.com/v1/archive'
WEATHER_CACHE = 'berlin_temperature_2015_2020.csv'
weather_query = {'latitude': 52.52, 'longitude': 13.41,
                 'start_date': '2015-01-01', 'end_date': '2020-09-30',
                 'hourly': 'temperature_2m', 'timezone': 'UTC'}
# Offline-safe temperature: use the local cache if it exists, otherwise call Open-Meteo
# once and cache the response so later (possibly offline) runs need no network. Ship the
# cache CSV alongside the notebook to make the SARIMAX and SVR sections fully reproducible.
if os.path.exists(WEATHER_CACHE):
    temp_hourly = pd.read_csv(WEATHER_CACHE, parse_dates=['time'], index_col='time')['temperature_2m']
    print(f'Loaded cached temperature from {WEATHER_CACHE}')
else:
    try:
        raw_weather = requests.get(ENDPOINT, params=weather_query, timeout=60).json()
    except Exception as exc:
        raise RuntimeError('Open-Meteo request failed and no local cache '
                           f'({WEATHER_CACHE}) was found. Connect once to build the '
                           'cache, or ship the cache CSV with the notebook.') from exc
    temp_hourly = pd.Series(raw_weather['hourly']['temperature_2m'],
                            index=pd.to_datetime(raw_weather['hourly']['time']),
                            name='temperature_2m')
    temp_hourly.index.name = 'time'
    temp_hourly.to_csv(WEATHER_CACHE)
    print(f'Fetched temperature from Open-Meteo and cached to {WEATHER_CACHE}')
# Normalise the index to UTC (a cache round-trip can drop or change the tz).
temp_hourly.index = pd.to_datetime(temp_hourly.index)
if temp_hourly.index.tz is None:
    temp_hourly.index = temp_hourly.index.tz_localize('UTC')
else:
    temp_hourly.index = temp_hourly.index.tz_convert('UTC')
temp_weekly = (temp_hourly.resample('W').mean()
               .reindex(demand_weekly.index).interpolate().bfill().ffill())
german_cal = holidays.Germany(years=range(2015, 2021))
def flag_holiday(week_ending):
    return int(any(d in german_cal for d in pd.date_range(end=week_ending, periods=7)))
holiday_weekly = pd.Series([flag_holiday(d) for d in demand_weekly.index], index=demand_weekly.index)
regs = pd.DataFrame({
    'temp': temp_weekly,
    'temp_sq': temp_weekly ** 2,
    'temp_prev': temp_weekly.shift(1).bfill(),
    'fest': holiday_weekly,
})
print('Regressors:', list(regs.columns), '| any nulls:', bool(regs.isna().any().any()))
regs_in = regs.iloc[:-TEST_WEEKS]
regs_out = regs.iloc[-TEST_WEEKS:]

In [ ]:
# temperature-load relationship shown as binned means with dispersion bars
bins = pd.cut(temp_weekly, 12)
grouped = demand_weekly.groupby(bins, observed=True)
centres = [iv.mid for iv in grouped.mean().index]

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(centres, grouped.mean().to_numpy(), yerr=grouped.std().to_numpy(),
            fmt='o', color=PAL['cyan'], ecolor=PAL['grey'], capsize=3, ms=6,
            label='Mean load +/- 1 sd per temperature bin')
smooth = np.polyfit(temp_weekly, demand_weekly, 2)
xs = np.linspace(temp_weekly.min(), temp_weekly.max(), 100)
ax.plot(xs, np.polyval(smooth, xs), color=PAL['pink'], lw=2.2, label='Quadratic fit')
ax.set_title('Weekly demand versus temperature')
ax.set_xlabel('Weekly mean temperature (C)')
ax.set_ylabel('Weekly mean load (MW)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Step 8 - SARIMAX with drivers

The chosen SARIMA structure is refitted with the driver matrix attached, producing a
conditional forecast.

In [ ]:
sarimax_res = SARIMAX(insample, exog=regs_in, order=main_order, seasonal_order=main_seasonal,
                      **SS_OFF).fit(disp=False, method='lbfgs', maxiter=420)
sarimax_out = sarimax_res.get_forecast(steps=TEST_WEEKS, exog=regs_out)
sarimax_fc = sarimax_out.predicted_mean
sarimax_fc.index = when
sarimax_pi = sarimax_out.conf_int(alpha=0.05)
sarimax_pi.index = when

sarimax_s = score(holdout, sarimax_fc)
print(f'SARIMAX RMSE={sarimax_s.rmse:.1f}  MAE={sarimax_s.mae:.1f}  MAPE={sarimax_s.mape:.2f}%')
print(f'SARIMA  RMSE={sarima_s.rmse:.1f}  (no drivers)')

fig, ax = plt.subplots(figsize=(12.5, 5))
ax.plot(when, holdout, color=PAL['navy'], lw=2.2, label='Observed')
ax.plot(when, sarimax_fc, color=PAL['mint'], lw=2.0, label='SARIMAX mean')
ax.fill_between(when, sarimax_pi.iloc[:, 0], sarimax_pi.iloc[:, 1],
                color=PAL['mint'], alpha=0.18, label='95% interval')
ax.set_title('SARIMAX (temperature + holiday) forecast')
ax.set_ylabel('MW')
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
sxr = trim_resid(sarimax_res, main_order, main_seasonal)
fig, ax = plt.subplots(figsize=(11.5, 3.6))
plot_acf(sxr, ax=ax, lags=52, color=PAL['mint'])
ax.set_title('SARIMAX residual ACF')
plt.tight_layout()
plt.show()
print('SARIMAX Ljung-Box:')
print(acorr_ljungbox(sxr, lags=[l for l in (10, 20, 52) if l < len(sxr)], return_df=True).to_string())

Adding the covariates shifts the SARIMAX RMSE relative to the plain SARIMA. The drivers
are only partly known ahead of time - temperature would come from a weather forecast in
deployment, while holidays are deterministic - so the figure remains a conditional one.

## Step 9 - Support Vector Regression: from features to forecast

The feature-based learner in this version is a **Support Vector Regression (SVR)** with a
radial-basis-function (RBF) kernel - a deliberate move away from the tree ensembles used
earlier (Random Forest in version 1, Gradient Boosting in version 2).

**Why a kernel method, and why SVR?** A support vector regressor fits the *flattest* function
that keeps most training points inside an epsilon-wide tube, balancing data fit against model
smoothness through a single penalty `C`. This margin / regularisation objective is a
fundamentally different inductive bias from the greedy, axis-aligned splits of a decision
tree: rather than carving the feature space into boxes, SVR searches for one smooth global
surface. For a signal like weekly demand - a smooth annual wave plus a smooth temperature
response - a smooth global hypothesis is a natural fit.

**Why the RBF kernel?** The kernel trick lets the model act as if the features were lifted
into a very high-dimensional space without ever constructing that space. The RBF (Gaussian)
kernel scores similarity by distance, so weeks with comparable season, temperature and recent
load receive comparable predictions. It represents non-linear effects - such as the U-shaped
temperature response - without us hand-coding them, and it is the standard, well-understood
default when no specific functional form is known in advance.

**Why not Random Forest or Gradient Boosting again?** Both are strong, but they belong to the
same tree-ensemble family and were already covered. Trees extrapolate as flat step functions
outside the training range and rank features natively; a kernel machine extrapolates smoothly
and needs permutation-based importance. Contrasting these families is exactly the kind of
alternative-solution comparison the assignment rewards.

**What SVR asks for in return** - all addressed in the stages below - is scaled inputs and
target, a smooth cyclical calendar encoding, a tuned `C`/`gamma`/`epsilon` triple, and
permutation importance in place of native coefficients.

In [ ]:
# a compact overview of the SVR pipeline used in this section
from matplotlib.patches import FancyBboxPatch

stages = ['Feature\nengineering', 'Standardise\nX and y', 'Grid search\n(TimeSeriesSplit)',
          'Fit tuned\nSVR (RBF)', 'Recursive\nforecast', 'Evaluate &\ndiagnose']
tints = [PAL['cyan'], PAL['mint'], PAL['gold'], PAL['pink'], PAL['navy'], PAL['grey']]

fig, ax = plt.subplots(figsize=(14, 2.3))
xs = np.linspace(0.5, 11.5, len(stages))
for x, label, tint in zip(xs, stages, tints):
    ax.add_patch(FancyBboxPatch((x - 0.85, 0.3), 1.7, 1.4, boxstyle='round,pad=0.1',
                                fc=tint, ec=PAL['navy'], alpha=0.75, lw=1.2))
    ax.text(x, 1.0, label, ha='center', va='center', fontsize=9, fontweight='bold',
            color='white' if tint == PAL['navy'] else PAL['navy'])
for x0, x1 in zip(xs[:-1], xs[1:]):
    ax.annotate('', xy=(x1 - 0.9, 1.0), xytext=(x0 + 0.9, 1.0),
                arrowprops=dict(arrowstyle='-|>', color=PAL['navy'], lw=1.6))
ax.set_xlim(-0.5, 12.5)
ax.set_ylim(0, 2)
ax.axis('off')
ax.set_title('Support Vector Regression workflow', fontweight='bold')
plt.tight_layout()
plt.show()

### 9.1 Feature engineering

The predictors combine recent-demand memory (`lag1`, `lag52`), temperature terms (`temp`,
`temp_sq`, `temp_prev`, a four-week trailing mean), a holiday flag and the calendar position
of the week. Every lag and rolling feature looks strictly backward (`shift(1)`, `shift(52)`,
a trailing mean), so no feature at week *t* uses information from week *t* or later. Two
design choices matter especially for a kernel model:

**Cyclical calendar encoding.** A raw week number (1...52) or month (1...12) misleads any
distance-based learner: numerically it places week 52 and week 1 fifty-one units apart when
they are in fact adjacent, so December and January land at opposite ends of the scale.
Encoding the week angle as a `(sin, cos)` pair wraps the calendar onto a circle, so late
December and early January become near neighbours - which is exactly how demand behaves. The
RBF kernel, judging similarity by distance, then treats consecutive weeks as genuinely close.

**Scaling.** Random Forest and Gradient Boosting are invariant to any monotone rescaling of a
feature, because their split rules only compare values. An RBF kernel is the opposite: it
works on Euclidean distances, so a feature measured in thousands of MW would swamp one
measured in tenths of a degree. We therefore standardise every predictor to zero mean and
unit variance, and standardise the target too so that the default `epsilon` tube and `C`
penalty sit on a sensible scale. All scalers are fit on the training window only and then
applied to the hold-out, so no future information leaks backward.

In [ ]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.inspection import permutation_importance

week_num = demand_weekly.index.isocalendar().week.astype(int).to_numpy()
dmatrix = pd.DataFrame(index=demand_weekly.index)
dmatrix['sin_yr'] = np.sin(2 * np.pi * week_num / 52)
dmatrix['cos_yr'] = np.cos(2 * np.pi * week_num / 52)
dmatrix['temp'] = regs['temp']
dmatrix['temp_sq'] = regs['temp'] ** 2
dmatrix['temp_prev'] = regs['temp'].shift(1)
dmatrix['temp_avg4'] = regs['temp'].rolling(4).mean()
dmatrix['lag1'] = demand_weekly.shift(1)
dmatrix['lag52'] = demand_weekly.shift(52)
dmatrix['fest'] = regs['fest']
dmatrix['target'] = demand_weekly.to_numpy()
dmatrix = dmatrix.dropna()

COLS = [c for c in dmatrix.columns if c != 'target']
dmatrix_in = dmatrix.iloc[:-TEST_WEEKS]
dmatrix_out = dmatrix.iloc[-TEST_WEEKS:]
X_in, y_in = dmatrix_in[COLS], dmatrix_in['target']
X_out, y_out = dmatrix_out[COLS], dmatrix_out['target']

# targets are standardized so that epsilon/C live on a sensible scale
y_scaler = StandardScaler()
y_in_z = y_scaler.fit_transform(y_in.to_numpy().reshape(-1, 1)).ravel()
print('SVR design matrix:', X_in.shape, '| features:', COLS)

### 9.2 A default baseline before tuning

Sound methodology fixes a reference point before optimising. We first fit an SVR with library
defaults inside the scaling pipeline and score it one step ahead on the hold-out. That number
tells us what an untuned kernel already achieves and gives the grid search a concrete target
to improve on.

In [ ]:
base_pipe = Pipeline([('scale', StandardScaler()), ('svr', SVR(kernel='rbf'))])
base_pipe.fit(X_in, y_in_z)
base_one = y_scaler.inverse_transform(base_pipe.predict(X_out).reshape(-1, 1)).ravel()
base_one_rmse = rmse_only(y_out, base_one)

default_svr = base_pipe.named_steps['svr']
print('Default SVR settings :', f'C={default_svr.C}, gamma="{default_svr.gamma}", epsilon={default_svr.epsilon}')
print(f'Default one-step RMSE : {base_one_rmse:.1f} MW  (reference for the tuned model)')

### 9.3 Hyperparameter optimisation

Three knobs govern an RBF SVR, each with a concrete meaning:

- **`C` - regularisation strength.** How heavily the model is penalised for points outside
  the epsilon tube. Small `C` keeps the surface very smooth (higher bias, lower variance);
  large `C` bends it to chase training points (lower bias, higher variance).
- **`gamma` - kernel reach.** The inverse width of each RBF bump. Small `gamma` lets a
  training point influence a wide neighbourhood (smoother, more global); large `gamma` makes
  its influence local and can over-fit. The `'scale'` option derives it from feature variance.
- **`epsilon` - the insensitive tube.** Errors below `epsilon` cost nothing, so it sets how
  tightly the tube hugs the data and how many points become support vectors.

We search these with `GridSearchCV` under a `TimeSeriesSplit`, so every validation fold lies
strictly in the future of its training fold - the only honest way to cross-validate a time
series. Read the winning triple through the descriptions above: a smaller `gamma` with a
moderate `C` points to a smooth, well-regularised surface, whereas a large `C` combined with a
large `gamma` would be a warning to check for over-fitting against the hold-out.

In [ ]:
svr_pipe = Pipeline([('scale', StandardScaler()), ('svr', SVR(kernel='rbf'))])
svr_space = {
    'svr__C': [1.0, 10.0, 40.0, 100.0],
    'svr__gamma': ['scale', 0.05, 0.1],
    'svr__epsilon': [0.05, 0.1, 0.2],
}
cv = TimeSeriesSplit(n_splits=4)
seeker = GridSearchCV(svr_pipe, svr_space, cv=cv,
                      scoring='neg_root_mean_squared_error', n_jobs=-1)
seeker.fit(X_in, y_in_z)
svr_best = seeker.best_estimator_
print('Best SVR settings:', seeker.best_params_)
print('CV RMSE (standardized target): {:.4f}'.format(-seeker.best_score_))
sv_used = svr_best.named_steps['svr'].support_.shape[0]
print(f'Support vectors retained: {sv_used} of {X_in.shape[0]} training weeks '
      f'({100 * sv_used / X_in.shape[0]:.0f}% of the sample).')

svr_grid_summary = pd.DataFrame({
    'Parameter': ['C', 'gamma', 'epsilon'],
    'Candidates tried': [str(svr_space['svr__C']), str(svr_space['svr__gamma']),
                         str(svr_space['svr__epsilon'])],
    'Selected': [seeker.best_params_['svr__C'], seeker.best_params_['svr__gamma'],
                 seeker.best_params_['svr__epsilon']],
})
print('\nHyperparameter search summary:')
print(svr_grid_summary.to_string(index=False))

The three figures above are read together. The **CV RMSE** comes from the inner time-series folds (on the standardised target) and gauges validation quality; the **hold-out** RMSE reported later is the honest out-of-sample number, and a wide gap between them would warn of over-fitting to the folds. The **baseline-versus-tuned** one-step comparison (printed with the forecasts below) shows whether the search paid off: a small drop means the RBF defaults were already well matched to the scaled features, a large drop means tuning materially helped. The **support-vector fraction** is a complexity gauge - a high fraction means most training weeks actively shape the fitted surface (a denser, more flexible model that can over-fit), while a low fraction means a sparse solution resting on a few decisive weeks.

### 9.4 Forecast generation

Two forecasts follow. The **one-step** version is handed the true previous week at every step -
a useful reference, but not a genuine two-year forecast. The **recursive** version is the
honest multi-step forecast: past the origin the model has no future actuals, so each
prediction becomes the lag input for the next week -

`predict(t)  ->  becomes lag-1 for week t+1  ->  predict(t+1)  ->  ...  repeat`

That is what makes the SVR comparable to SARIMA across the whole horizon. It is also why
recursive error *accumulates*: in one-step mode every prediction is re-anchored to a fresh
true observation, so mistakes cannot build up, whereas in recursive mode the model feeds on
its own imperfect outputs, so a small systematic bias is amplified week after week. Some
recursive drift is therefore expected and is examined in the diagnostics that follow.

In [ ]:
# (a) one-step reference: actual previous-week load feeds the lag features.
onestep_z = svr_best.predict(X_out)
svr_onestep = pd.Series(y_scaler.inverse_transform(onestep_z.reshape(-1, 1)).ravel(),
                        index=y_out.index)

# (b) recursive multi-step: the model's own outputs refill the load lags past the origin,
#     giving a genuine 104-week forecast comparable to SARIMA. Weather/holiday stay observed.
axis = demand_weekly.index
origin = insample.size
memory = list(demand_weekly.iloc[:origin].to_numpy())
temp_line = regs['temp']
fest_line = regs['fest']

collected = []
for k in range(TEST_WEEKS):
    j = origin + k
    stamp = axis[j]
    wnum = int(stamp.isocalendar()[1])
    nxt_feats = {
        'sin_yr': np.sin(2 * np.pi * wnum / 52),
        'cos_yr': np.cos(2 * np.pi * wnum / 52),
        'temp': temp_line.iloc[j],
        'temp_sq': temp_line.iloc[j] ** 2,
        'temp_prev': temp_line.iloc[j - 1],
        'temp_avg4': temp_line.iloc[j - 3:j + 1].mean(),
        'lag1': memory[j - 1],
        'lag52': memory[j - 52],
        'fest': fest_line.iloc[j],
    }
    z = svr_best.predict(pd.DataFrame([nxt_feats])[COLS])[0]
    value = float(y_scaler.inverse_transform([[z]])[0, 0])
    collected.append(value)
    memory.append(value)
svr_recursive = pd.Series(collected, index=when)

svr_one_s = score(y_out, svr_onestep)
svr_rec_s = score(holdout, svr_recursive)
print(f'SVR one-step   RMSE={svr_one_s.rmse:8.1f}  MAE={svr_one_s.mae:8.1f}  MAPE={svr_one_s.mape:5.2f}%  (actual lag-1)')
print(f'SVR recursive  RMSE={svr_rec_s.rmse:8.1f}  MAE={svr_rec_s.mae:8.1f}  MAPE={svr_rec_s.mape:5.2f}%  (true 2-yr)')
print(f'Tuning moved the one-step RMSE from {base_one_rmse:.1f} MW (default) '
      f'to {svr_one_s.rmse:.1f} MW (tuned).')

In [ ]:
fig, ax = plt.subplots(figsize=(12.5, 5))
ax.plot(dmatrix_in.index[-70:], y_in.iloc[-70:], color=PAL['grey'], lw=1.1, label='Recent history')
ax.plot(when, holdout, color=PAL['navy'], lw=2.2, label='Observed')
ax.plot(when, svr_recursive, color=PAL['pink'], lw=2.0, label='SVR recursive (multi-step)')
ax.plot(when, svr_onestep, color=PAL['gold'], lw=1.4, ls='--', label='SVR one-step (actual lag-1)')
ax.set_title('Support Vector Regression forecasts')
ax.set_ylabel('MW')
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

### 9.5 Evaluation and diagnostics

A kernel SVR exposes no regression coefficients and no built-in feature ranking, so each
predictor's contribution is measured by **permutation importance**: after training, a feature's
values are randomly shuffled on the hold-out and the resulting rise in RMSE is recorded. A large
rise means the model genuinely leaned on that feature. Repeating the shuffle many times gives
the spread drawn as boxes below.

In [ ]:
# kernel machines expose no native importance -> permutation importance on the hold-out
y_out_z = y_scaler.transform(y_out.to_numpy().reshape(-1, 1)).ravel()
perm = permutation_importance(svr_best, X_out, y_out_z, n_repeats=25,
                              random_state=RNG, scoring='neg_root_mean_squared_error')
order = perm.importances_mean.argsort()

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.boxplot(perm.importances[order].T, vert=False, labels=np.array(COLS)[order],
           patch_artist=True,
           boxprops=dict(facecolor=PAL['cyan'], alpha=0.6),
           medianprops=dict(color=PAL['pink']))
ax.set_title('SVR permutation importance (hold-out)')
ax.set_xlabel('Increase in RMSE when the feature is shuffled')
plt.tight_layout()
plt.show()

In [ ]:
import scipy.stats as sstats

svr_err = holdout - svr_recursive          # residuals of the true multi-step forecast

fig, (axq, axsc) = plt.subplots(1, 2, figsize=(13, 4.3))
sstats.probplot(svr_err, dist='norm', plot=axq)
axq.get_lines()[0].set(color=PAL['cyan'], markersize=4)
axq.get_lines()[1].set(color=PAL['pink'])
axq.set_title('SVR residuals - normal Q-Q')

axsc.scatter(svr_recursive, svr_err, color=PAL['cyan'], alpha=0.7, edgecolor=PAL['navy'], s=28)
axsc.axhline(0, color=PAL['pink'], lw=1.4)
axsc.set_title('Residual versus prediction')
axsc.set_xlabel('Predicted load (MW)')
axsc.set_ylabel('Residual = observed - predicted (MW)')
plt.tight_layout()
plt.show()

In [ ]:
# where do the errors concentrate?
abs_err = svr_err.abs()
season_label = np.where(np.isin(when.month, [12, 1, 2]), 'winter',
                        np.where(np.isin(when.month, [6, 7, 8]), 'summer', 'spring/autumn'))
season_mae = pd.Series(abs_err.to_numpy(), index=season_label).groupby(level=0).mean().round(0)

hol_mask = holiday_weekly.reindex(when).fillna(0).astype(bool).to_numpy()
hol_mae = abs_err.to_numpy()[hol_mask].mean() if hol_mask.any() else float('nan')
plain_mae = abs_err.to_numpy()[~hol_mask].mean()

print('Mean absolute error by season (MW):')
print(season_mae.to_string())
print(f'\nHoliday-week MAE     : {hol_mae:8.0f} MW')
print(f'Non-holiday-week MAE : {plain_mae:8.0f} MW')
print('\nSix largest weekly forecast errors:')
worst = svr_err.reindex(abs_err.sort_values(ascending=False).index).head(6)
for ts, e in worst.items():
    print(f'  {ts.date()}   {e:+8.0f} MW')

### 9.6 Interpreting the SVR results

**What drives the forecast.** The permutation ranking is led by the recent-load and year-ago
lags, echoing the strong annual persistence in the STL seasonal term; the cyclical calendar
pair and temperature refine the fit rather than lead it. The Q-Q plot and the
residual-versus-prediction scatter check the error behaviour: departures in the Q-Q tails flag
heavy-tailed weeks, while any funnel shape in the scatter would signal heteroscedasticity
(errors that grow at particular demand levels). The seasonal split usually exposes the hardest
stretches - deep-winter peaks and the 2020 spring dip - where every model here struggles.

**Kernel learning versus tree ensembles.** The three feature-based models sit in different
families and fail in different ways. Random Forest (bagging) averages many de-correlated trees
- robust, but it extrapolates as a flat line. Gradient Boosting (boosting) fits residuals stage
by stage and is often the sharpest in-range. SVR (kernel) fits one smooth global surface and
extrapolates smoothly. On a series ruled by a repeatable annual cycle the smooth kernel fit is
competitive in-range, and over the long recursive horizon it tends to drift gently rather than
lock onto a single learned level.

**Weaknesses to keep in view.** SVR is sensitive to feature scaling (handled here); its
`C`/`gamma`/`epsilon` search costs more than fitting a single tree ensemble; training scales
super-linearly with sample size (harmless at ~200 weeks but limiting on hourly data); it gives
no native prediction interval; and, like every model here, it is exposed to recursive drift over
the two-year horizon.

**If the SVR trails the tree models or the seasonal replay, that is an acceptable and useful
result.** It would say that the lag interactions in this series are captured at least as well by
axis-aligned tree splits, or simply by repeating last year, and that the extra machinery of a
kernel does not pay off here. We report the outcome as measured rather than tuning until the
kernel "wins" - the cross-family comparison, not a leaderboard position, is the point.

## Step 10 - Hourly LSTM
### Literature review
The Long Short-Term Memory cell (Hochreiter & Schmidhuber, 1997) adds input, forget and
output gates to a recurrent network, letting it carry information across long spans
without the vanishing gradients that hamper plain RNNs. That property has made LSTMs a
mainstay of short-term electricity-load forecasting. Kong et al. (2019, *IEEE Transactions
on Smart Grid*) report an LSTM beating classical baselines on residential load by learning
short-run dynamics and daily/seasonal shape directly from the raw signal, without the
stationarity assumptions SARIMA relies on. Marino et al. (2016, *IECON*) apply standard and
sequence-to-sequence LSTMs to building-level demand; Shi et al. (2018, *IEEE Transactions on
Smart Grid*) tackle the high volatility of individual households with a pooling-based deep
RNN that curbs over-fitting; and Bouktif et al. (2018, *Energies*) tune LSTM depth and
look-back via feature selection and a genetic algorithm, improving on shallower models. The
recurring caveats across this literature are a hunger for data and error accumulation in
long recursive forecasts - both of which we quantify below by contrasting a rolling
one-step evaluation with a genuine open-loop multi-step forecast.
Sequences are built from the hourly series with a 168-hour (one-week) look-back using an
efficient sliding-window view.

**References**
- Hochreiter, S. & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation*, 9(8), 1735-1780.
- Kong, W., Dong, Z. Y., Jia, Y., Hill, D. J., Xu, Y. & Zhang, Y. (2019). Short-term residential load forecasting based on LSTM recurrent neural network. *IEEE Transactions on Smart Grid*, 10(1), 841-851.
- Marino, D. L., Amarasinghe, K. & Manic, M. (2016). Building energy load forecasting using deep neural networks. *IECON 2016 - 42nd Annual Conference of the IEEE Industrial Electronics Society*, 7046-7051.
- Shi, H., Xu, M. & Li, R. (2018). Deep learning for household load forecasting - a novel pooling deep RNN. *IEEE Transactions on Smart Grid*, 9(5), 5271-5280.
- Bouktif, S., Fiaz, A., Ouni, A. & Serhani, M. A. (2018). Optimal deep learning LSTM model for electric load forecasting using feature selection and genetic algorithm. *Energies*, 11(7), 1636.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
from numpy.lib.stride_tricks import sliding_window_view

mw_hourly = power_60min['mw'].copy()
TEST_HRS = 24 * 7 * 52 * 2
train_hours = len(mw_hourly) - TEST_HRS  # boundary of the two-year test window
minmax = MinMaxScaler()
# Fit the scaler on the TRAINING hours only, then transform the whole series
# (fitting on all hours would leak the test-window min/max into the inputs).
minmax.fit(mw_hourly.to_numpy()[:train_hours].reshape(-1, 1))
mw_norm = minmax.transform(mw_hourly.to_numpy().reshape(-1, 1)).ravel()

LOOKBACK = 168
frames = sliding_window_view(mw_norm, LOOKBACK)
X_all = frames[:-1][:, :, None]
y_all = mw_norm[LOOKBACK:]

edge = len(X_all) - TEST_HRS
X_learn, y_learn = X_all[:edge], y_all[:edge]
X_check, y_check = X_all[edge:], y_all[edge:]
print('LSTM tensors -> learn', X_learn.shape, '| check', X_check.shape)

### Architecture and hyperparameter sweep

Five recurrent designs are compared - varying width, depth, dropout and batch size - each
trained briefly with early stopping. The configuration with the lowest hourly hold-out RMSE
is retrained and carried forward.

In [ ]:
def build_rnn(widths, drop_rate):
    seq = tf.keras.Sequential()
    seq.add(Input(shape=(LOOKBACK, 1)))
    for pos, w in enumerate(widths):
        seq.add(LSTM(w, return_sequences=(pos < len(widths) - 1)))
        seq.add(Dropout(drop_rate))
    seq.add(Dense(20, activation='relu'))
    seq.add(Dense(1, activation='linear'))
    seq.compile(optimizer='adam', loss='mean_squared_error')
    return seq


designs = [
    {'name': 'single-40', 'widths': [40], 'drop': 0.15, 'batch': 128},
    {'name': 'pair-64-32', 'widths': [64, 32], 'drop': 0.2, 'batch': 128},
    {'name': 'pair-100-50', 'widths': [100, 50], 'drop': 0.3, 'batch': 64},
    {'name': 'single-80', 'widths': [80], 'drop': 0.25, 'batch': 96},
    {'name': 'trio-100-60-30', 'widths': [100, 60, 30], 'drop': 0.3, 'batch': 64},
]

board = []
for d in designs:
    tf.random.set_seed(RNG)
    net = build_rnn(d['widths'], d['drop'])
    stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    hist = net.fit(X_learn, y_learn, epochs=10, batch_size=d['batch'],
                   validation_split=0.1, callbacks=[stop], verbose=0)
    pred = minmax.inverse_transform(net.predict(X_check, verbose=0)).ravel()
    truth = minmax.inverse_transform(y_check.reshape(-1, 1)).ravel()
    board.append({'design': d['name'], 'widths': '-'.join(map(str, d['widths'])),
                  'dropout': d['drop'], 'batch': d['batch'],
                  'val_loss': round(min(hist.history['val_loss']), 6),
                  'hourly_rmse': round(rmse_only(truth, pred), 2)})
board_tbl = pd.DataFrame(board).sort_values('hourly_rmse').reset_index(drop=True)
print(board_tbl.to_string(index=False))
champion = next(d for d in designs if d['name'] == board_tbl.iloc[0]['design'])
print('\nChampion design:', champion['name'])

In [ ]:
tf.random.set_seed(RNG)
rnn = build_rnn(champion['widths'], champion['drop'])
stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
curve = rnn.fit(X_learn, y_learn, epochs=10, batch_size=champion['batch'],
                validation_split=0.1, callbacks=[stop], verbose=0)
print('Final network retrained:', champion['name'])

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.2))
ax.plot(curve.history['loss'], color=PAL['cyan'], marker='o', ms=3, label='Train')
ax.plot(curve.history['val_loss'], color=PAL['pink'], marker='D', ms=3, label='Validation')
ax.set_title('LSTM learning curve')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE (scaled)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### Rolling versus open-loop error

The **rolling** one-step forecast is fed the true previous 168 hours at each step; the
**open-loop** forecast recycles its own predictions, which is the only honest way to cover
the entire two-year span from one origin. Both are reported, and the two are not
interchangeable.

In [ ]:
# rolling one-step: each prediction sees the genuine previous week of hours
roll_pred = minmax.inverse_transform(rnn.predict(X_check, batch_size=256, verbose=0)).ravel()
roll_truth = minmax.inverse_transform(y_check.reshape(-1, 1)).ravel()

# open-loop recursion: feed predictions back into the window
buffer = mw_norm[edge:edge + LOOKBACK].copy()
loop_norm = []
for stepi in range(TEST_HRS):
    nxt = rnn.predict(buffer.reshape(1, LOOKBACK, 1), verbose=0)[0, 0]
    loop_norm.append(nxt)
    buffer = np.roll(buffer, -1)
    buffer[-1] = nxt
    if (stepi + 1) % 2000 == 0:
        print(f'   open-loop {stepi + 1}/{TEST_HRS}')
loop_pred = minmax.inverse_transform(np.array(loop_norm).reshape(-1, 1)).ravel()

stamp_hr = mw_hourly.index[LOOKBACK + edge:]
roll_hr_rmse = rmse_only(roll_truth, roll_pred)
loop_hr_rmse = rmse_only(roll_truth[:TEST_HRS], loop_pred)

roll_wk = pd.Series(roll_pred, index=stamp_hr).resample('W').mean()
truth_wk = pd.Series(roll_truth, index=stamp_hr).resample('W').mean()
lstm_roll_wk = rmse_only(truth_wk, roll_wk)

loop_wk = pd.Series(loop_pred, index=stamp_hr[:TEST_HRS]).resample('W').mean()
loop_truth_wk = pd.Series(roll_truth[:TEST_HRS], index=stamp_hr[:TEST_HRS]).resample('W').mean()
lstm_loop_wk = rmse_only(loop_truth_wk, loop_wk)

print(f'Rolling   hourly RMSE : {roll_hr_rmse:8.1f} MW  [sees actual history]')
print(f'Rolling   weekly RMSE : {lstm_roll_wk:8.1f} MW  [sees actual history]')
print(f'Open-loop weekly RMSE : {lstm_loop_wk:8.1f} MW  [true multi-step]')
print(f'Open-loop hourly RMSE : {loop_hr_rmse:8.1f} MW  [true multi-step]')

Once labelled correctly, the rolling weekly RMSE falls below the hourly figure because
averaging cancels independent hourly errors, whereas the open-loop weekly RMSE is far larger:
feedback compounds error over two years, so a univariate open-loop LSTM is not viable at this
horizon without external drivers.

## Step 11 - Model scorecard and comparison

All models are gathered with a forecast-type tag so that like is compared with like. The
diverging bar chart shows each multi-step model's RMSE *relative to the seasonal-replay
benchmark* - bars to the left are better than the benchmark, bars to the right are worse.

In [ ]:
def scorecard_row(tag, actual, guess, kind):
    s = score(actual, guess)
    return {'Model': tag, 'Kind': kind, 'RMSE': round(s.rmse, 1),
            'MAE': round(s.mae, 1), 'MAPE': round(s.mape, 2)}


scorecard = pd.DataFrame([
    scorecard_row('Average', holdout, avg_fc, 'multi-step'),
    scorecard_row('Last value', holdout, hold_fc, 'multi-step'),
    scorecard_row('Seasonal replay', holdout, snaive_fc, 'multi-step'),
    scorecard_row('Drift', holdout, drift_fc, 'multi-step'),
    scorecard_row('SARIMA', holdout, sarima_fc, 'multi-step'),
    scorecard_row('SARIMAX', holdout, sarimax_fc, 'multi-step (cond.)'),
    scorecard_row('SVR recursive', holdout, svr_recursive, 'multi-step (cond.)'),
    scorecard_row('SVR one-step', y_out, svr_onestep, 'one-step (lag-1)'),
    scorecard_row('LSTM open-loop', loop_truth_wk, loop_wk, 'multi-step'),
    scorecard_row('LSTM rolling', truth_wk, roll_wk, 'one-step (lag-1)'),
]).sort_values('RMSE').reset_index(drop=True)
bench_rmse = scorecard.loc[scorecard['Model'] == 'Seasonal replay', 'RMSE'].iloc[0]
scorecard['vs benchmark'] = (scorecard['RMSE'] - bench_rmse).round(1)
print('Seasonal-replay RMSE = {} MW\n'.format(bench_rmse))
print(scorecard.to_string(index=False))

In [ ]:
ms = scorecard[scorecard['Kind'].str.startswith('multi-step')].copy()
ms = ms.sort_values('vs benchmark')
bar_cols = [PAL['mint'] if v <= 0 else PAL['pink'] for v in ms['vs benchmark']]

fig, ax = plt.subplots(figsize=(10.5, 5))
ax.barh(ms['Model'], ms['vs benchmark'], color=bar_cols, edgecolor=PAL['navy'])
ax.axvline(0, color=PAL['navy'], lw=1.3)
for y, v in zip(range(len(ms)), ms['vs benchmark']):
    ax.text(v, y, f' {v:+.0f}', va='center',
            ha='left' if v >= 0 else 'right', fontsize=8)
ax.set_title('Multi-step RMSE relative to seasonal replay (left = better)')
ax.set_xlabel('RMSE difference vs benchmark (MW)')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15.5, 6.2))
ax.plot(insample.index, insample, color=PAL['grey'], lw=0.8, alpha=0.7, label='Train')
ax.plot(when, holdout, color=PAL['navy'], lw=2.4, label='Observed')
ax.plot(when, snaive_fc, color=PAL['gold'], lw=1.4, ls=(0, (5, 2)), label='Seasonal replay')
ax.plot(when, sarima_fc, color=PAL['cyan'], lw=1.4, label='SARIMA')
ax.plot(when, sarimax_fc, color=PAL['mint'], lw=1.4, label='SARIMAX')
ax.plot(when, svr_recursive, color=PAL['pink'], lw=2.2, label='SVR recursive')
lo_y = min(insample.min(), holdout.min()) * 0.9
hi_y = max(insample.max(), holdout.max()) * 1.1
ax.set_ylim(lo_y, hi_y)
ax.set_title('Multi-step forecasts against observed demand (open-loop LSTM shown apart)')
ax.set_ylabel('MW')
ax.legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(15, 3.8))
ax.plot(when, holdout, color=PAL['navy'], lw=2.0, label='Observed')
ax.plot(loop_wk.index, loop_wk, color=PAL['pink'], lw=1.5, label='LSTM open-loop')
ax.set_title('Open-loop LSTM drift across the two-year horizon')
ax.set_ylabel('MW')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Step 12 - Analytical questions

**Q1 - Which models meaningfully beat the seasonal-replay benchmark?**
Judged only within a forecast type, the recursive SVR is the only multi-step model that
comes level with the seasonal replay - at best matching it within run-to-run noise - while
SARIMAX, SARIMA and the naive baselines trail it and the open-loop LSTM is far adrift
through recursive drift. No multi-step model improves on 'repeat last year' by a margin
large enough to be decisive. German weekly
demand is governed by a near-constant annual cycle, so replaying last year is an exceptionally
strong two-year baseline. The hold-out also contains the 2020 COVID-19 dip - an exogenous
shock none of the models see coming - which further protects the naive replay. The one-step
SVR and rolling LSTM look sharper only because they read the true previous value and are not
comparable to the multi-step forecasts.

**Q2 - How was leakage avoided in the temperature features?**
All engineered features look strictly backward: `temp_prev = shift(1)`, `temp_avg4` is a
trailing four-week mean, `lag1 = shift(1)` and `lag52 = shift(52)`. Nothing at week *t* draws
on week *t* or later. The current temperature and holiday flag are used only under the stated
conditional assumption, and the recursive SVR forecast refills its own load lags with
predictions once past the origin, so no future actual leaks into the multi-step figures.

**Q3 - Justify the differencing orders and seasonal period.**
The non-seasonal `d` is not hard-coded. Because AIC is not comparable across different `d`,
the search refits candidate `(p, q)` orders at both `d = 0` and `d = 1`, then selects the
*smallest* `d` that yields at least one model with white-noise residuals (Ljung-Box
`p > 0.05`). This respects the stationarity evidence - ADF rejects a unit root at the level
and KPSS also reads the level as broadly stationary, which argues for a small `d` - while
refusing a low-AIC order that leaves residual autocorrelation; `d = 2` is rejected as
over-differencing. `D = 1` at `s = 52`: STL and the ACF confirm a dominant annual cycle in
weekly data, removed by one seasonal difference at lag 52. Within the chosen `d`, the
ordinary `(p, q)` and the seasonal `(P, Q)` over `{0, 1}` are ranked by AIC among the
residual-adequate candidates, with a parsimony tie-break (Burnham & Anderson).

**Q4 - Do the covariates help, and are they known at the origin?**
Temperature (with a squared term for the U-shaped response), its lag and the holiday flag move
the SARIMAX RMSE relative to plain SARIMA. They are only partly known ahead of time -
temperature needs a forecast, holidays are deterministic - so the result is conditional.

**Q5 - Interpretability and complexity.**

| Aspect | SARIMAX | SVR (RBF) | LSTM |
|---|---|---|---|
| Interpretability | High (coefficients, intervals) | Low-medium (permutation only) | Low (black box) |
| Complexity | Low-medium | Medium (scaling + kernel tuning) | High (architecture + GPU) |
| Uncertainty | Native intervals | None natively | None natively |
| Non-linearity | Manual (squared term) | Native (kernel) | Native (learned) |
| Training cost | Minutes | Seconds to minutes | Minutes to hours |

**Q6 - Which model for operational use?**
**SARIMAX** remains the pragmatic operational choice. It is not the most accurate here - the
seasonal replay and recursive SVR match or beat it - but it uniquely combines native
prediction intervals for risk planning, interpretable temperature/holiday coefficients for
scenario analysis, direct support for exogenous drivers and cheap retraining. The SVR is a
capable non-linear competitor but offers no built-in uncertainty and less transparency; the
open-loop LSTM is unsuited to this long horizon. A deployment should keep benchmarking against
the seasonal replay and retrain as new, post-shock data arrives.